# Ticket Classification with LLM

Notebook to classify support tickets using an LLM (OpenAI).
Steps: load config -> build prompt template -> validate input -> call LLM -> print output.

In [50]:
import os
import json

from dotenv import load_dotenv

from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage

# Load environment variables
load_dotenv()

MODEL_NAME = os.getenv("OPENAI_MODEL", "gpt-4o-mini")

llm = ChatOpenAI(
    model=MODEL_NAME,
    temperature=0
)

## Prompt Template

Support ticket classification system prompt: categories, team mapping, priority rules, and output format.

In [51]:
SYSTEM_MESSAGE = """You are an AI support ticket classifier for an e-commerce marketplace.
Your job is NOT to resolve the customer's issue or generate a reply — only to
read their message and produce a structured routing decision.
The customer is always the BUYER, never the seller or merchant.

CATEGORIES (classify into exactly one)
- order_issue            : order not placed, cancelled, incorrect, or needs modification
- delivery_logistics     : delayed, stuck, or marked delivered but not received; delivery agent issues
- returns_refunds        : return, replacement, or exchange requests; pickup failure; refund pending/incorrect
- payments_billing       : payment failure, duplicate charge, EMI issue, invoice/GST, subscription billing
- product_issue          : damaged, defective, incomplete, or poor-quality product
- account_access         : login, password reset, OTP problems, account locked
- wallet_offers          : wallet balance, cashback, coupons, gift cards, promotions
- seller_related         : seller misconduct, seller information issues, marketplace trust
- warranty_installation  : warranty claims, installation delays or requests
- technical_platform     : website/app bugs not caused by a third-party service
- fraud_security         : unauthorized transactions, phishing, suspicious OTP requests, account compromise
- feedback_complaints    : general complaints, poor service, escalation of a known issue
- general_inquiry        : general questions, or messages with too little information to classify confidently

CATEGORY -> ASSIGNED TEAM (use only this mapping, never invent a team)
order_issue -> Order Management
delivery_logistics -> Logistics & Delivery Ops
returns_refunds -> Returns & Refunds Team
payments_billing -> Payments & Billing
product_issue -> Product Quality Support
account_access -> Account & Identity Support
wallet_offers -> Rewards & Promotions
seller_related -> Marketplace Trust
warranty_installation -> Warranty & Installation
technical_platform -> Platform Engineering
fraud_security -> Trust & Security
feedback_complaints -> Customer Experience
general_inquiry -> Customer Support Desk

PRIORITY
High   - financial loss already happened; fraud/security concern; customer fully
         blocked; order shows delivered but not received; safety-related product issue
Medium - valid issue needing action, affecting a single order/account, with other
         services still usable
Low    - general questions, minor inconvenience, or insufficient information
         (general_inquiry)

RULES
1. Classify by the customer's PRIMARY REQUEST, not just the underlying cause,
   and pick a single category even if more than one could apply.
   e.g. "Phone is damaged" -> product_issue, but "I want to return my damaged
   phone" -> returns_refunds. Mention a genuinely relevant second category
   briefly in the reasoning, never as the chosen category.
2. Tone never changes the category. Anger, all-caps, or exclamation marks may
   raise the priority but never the category.
3. Any mention of suspicious calls, OTP requests, phishing, unauthorized
   transactions, or account compromise is ALWAYS category = fraud_security,
   priority = High, even if the customer sounds unsure.
4. If the message is too vague to classify with confidence (e.g. "broken",
   "not working", "help"), do not guess — use general_inquiry and state in the
   reasoning what information is missing. A missing order ID, product name, or
   customer ID alone should NOT trigger general_inquiry if the intent is
   otherwise clear.
5. Empty, meaningless, or unreadable input -> general_inquiry, priority Low.
6. Classify correctly regardless of input language; reasoning is always in
   English.
7. Identical inputs must always produce the same classification, priority, and
   team — never alternate between equally valid options.

OUTPUT
Return exactly one JSON object and nothing else (no markdown, no extra text,
no extra or missing attributes):
{
  "category": "<one of the 13 category values above>",
  "priority": "High" | "Medium" | "Low",
  "assigned_team": "<team from the mapping above>",
  "reasoning": "<one concise English sentence: why this category, why this priority>"
}

EXAMPLES

Input: "My payment failed twice while checking out but the amount was deducted both times."
Output: {"category": "payments_billing", "priority": "High", "assigned_team": "Payments & Billing", "reasoning": "Duplicate payment deduction caused direct financial loss."}

Input: "THIS IS RIDICULOUS. My order shows delivered but I never received it."
Output: {"category": "delivery_logistics", "priority": "High", "assigned_team": "Logistics & Delivery Ops", "reasoning": "Order marked delivered but not received; needs urgent logistics investigation."}

Input: "broken"
Output: {"category": "general_inquiry", "priority": "Low", "assigned_team": "Customer Support Desk", "reasoning": "Message lacks any detail about what product or service is affected."}

Input: "I want to return this phone because the screen is cracked."
Output: {"category": "returns_refunds", "priority": "Medium", "assigned_team": "Returns & Refunds Team", "reasoning": "Primary request is a return for a damaged product."}

Input: "Someone claiming to be customer support asked for my OTP."
Output: {"category": "fraud_security", "priority": "High", "assigned_team": "Trust & Security", "reasoning": "Reported phishing attempt requesting OTP, treated as high priority despite uncertainty."}
"""


def build_prompt(ticket_text: str) -> str:
    return f'Input: "{ticket_text}"\nOutput:'

## Input Preprocessing (Summarization)

Long tickets are summarized before classification to keep the prompt short and
cheap. We measure length in tokens (tiktoken); tickets at or under the
threshold pass through unchanged, longer ones go through a cheap LLM call
(`gpt-4o-mini`) that condenses them while preserving the customer's issue and
key details.

In [52]:
import tiktoken

TOKEN_THRESHOLD = 300  # tickets at or under this token count pass through unchanged
SUMMARY_MODEL_NAME = "gpt-4o-mini"

summarizer_llm = ChatOpenAI(
    model=SUMMARY_MODEL_NAME,
    temperature=0
)

_tokenizer = tiktoken.encoding_for_model("gpt-4o-mini")

SUMMARIZE_SYSTEM_MESSAGE = """You are preparing a long customer support ticket for an AI routing system.
Condense the message, preserving ONLY the following if present:
- The primary issue
- The customer's requested action
- Financial loss or incorrect charges
- Fraud or security concerns
- Delivery or order status
- Refund or payment details
- Product defects or damage
- Urgency or emotional tone (e.g. anger, all-caps, exclamation marks, repeated escalation)
- Order ID, product name, or account identifiers, if mentioned

Remove:
- Greetings and sign-offs
- Repeated or duplicate complaints
- Email signatures
- Quoted previous agent replies or email threads

Rules:
- Do not infer, assume, or add any information not present in the original message.
- Keep the result as short as possible while preserving the details above —
  a few sentences is usually enough.
- Output only the condensed ticket text itself. No labels, no preamble, no
  explanation."""


def count_tokens(text: str) -> int:
    return len(_tokenizer.encode(text))


def summarize_ticket(ticket_text: str) -> str:
    """Summarize a long ticket using a cheap LLM call before classification."""
    messages = [
        SystemMessage(content=SUMMARIZE_SYSTEM_MESSAGE),
        HumanMessage(content=ticket_text),
    ]
    response = summarizer_llm.invoke(messages)
    return response.content.strip()


def preprocess_ticket_text(ticket_text: str) -> str:
    """Pass short tickets through unchanged; summarize tickets over TOKEN_THRESHOLD tokens."""
    if count_tokens(ticket_text) <= TOKEN_THRESHOLD:
        return ticket_text
    return summarize_ticket(ticket_text)

## Input Validation

Check the input isn't blank before calling the LLM.

In [53]:
def validate_input(ticket: str) -> str:
    """
    Validate and normalize the incoming support ticket.
    """

    if ticket is None or not ticket.strip():
        raise ValueError(
            "Input ticket text is blank. Please provide a valid ticket."
        )

    clean_ticket = ticket.strip()
    return preprocess_ticket_text(clean_ticket)

## Call the LLM

In [54]:
def build_messages(ticket: str):

    return [
        SystemMessage(content=SYSTEM_MESSAGE),
        HumanMessage(
            content=f'Input: "{ticket}"\nOutput:'
        )
    ]

In [55]:
def classify_ticket(ticket: str) -> dict:

    clean_ticket = validate_input(ticket)

    messages = build_messages(clean_ticket)

    response = llm.invoke(messages)

    return json.loads(response.content)

In [56]:
sample_ticket = "My login keeps failing after the last update."

result = classify_ticket(sample_ticket)

print(json.dumps(result, indent=4))

{
    "category": "account_access",
    "priority": "Medium",
    "assigned_team": "Account & Identity Support",
    "reasoning": "Customer is experiencing login issues, which affects account access but does not block all services."
}


## Output Validation (Pydantic) & Retry

We never trust the raw LLM output directly. It is parsed and validated against a strict
Pydantic schema (4 required fields, `category`/`priority` restricted to known values,
`assigned_team` cross-checked against the category -> team mapping).

- Attempt 1: call the LLM, validate the result.
- If validation fails, attempt 2: call the LLM again with the validation error fed back
  so it can correct itself.
- If attempt 2 also fails, fall back to a fixed `general_inquiry` / `Medium` priority
  response so the pipeline never crashes or routes on bad data.

In [57]:
from typing import Literal
from pydantic import BaseModel, ValidationError, field_validator

CATEGORY_TEAM_MAP = {
    "order_issue": "Order Management",
    "delivery_logistics": "Logistics & Delivery Ops",
    "returns_refunds": "Returns & Refunds Team",
    "payments_billing": "Payments & Billing",
    "product_issue": "Product Quality Support",
    "account_access": "Account & Identity Support",
    "wallet_offers": "Rewards & Promotions",
    "seller_related": "Marketplace Trust",
    "warranty_installation": "Warranty & Installation",
    "technical_platform": "Platform Engineering",
    "fraud_security": "Trust & Security",
    "feedback_complaints": "Customer Experience",
    "general_inquiry": "Customer Support Desk",
}

FALLBACK_RESULT = {
    "category": "general_inquiry",
    "priority": "Medium",
    "assigned_team": "Customer Support Desk",
    "reasoning": "Automated classification failed validation; routed to general queue for manual review.",
}


class TicketClassification(BaseModel):
    category: Literal[tuple(CATEGORY_TEAM_MAP.keys())]
    priority: Literal["High", "Medium", "Low"]
    assigned_team: str
    reasoning: str

    @field_validator("assigned_team")
    @classmethod
    def team_matches_category(cls, assigned_team, info):
        category = info.data.get("category")
        expected_team = CATEGORY_TEAM_MAP.get(category)
        if expected_team is not None and assigned_team != expected_team:
            raise ValueError(
                f"assigned_team '{assigned_team}' does not match the required team "
                f"'{expected_team}' for category '{category}'"
            )
        return assigned_team


def parse_and_validate(raw_output: str) -> TicketClassification:
    data = json.loads(raw_output)
    return TicketClassification(**data)

In [58]:
from langchain_core.messages import AIMessage


def robust_classify_ticket(ticket: str) -> dict:
    clean_ticket = validate_input(ticket)
    messages = build_messages(clean_ticket)

    # Attempt 1
    raw_output = llm.invoke(messages).content
    try:
        return parse_and_validate(raw_output).model_dump()
    except (json.JSONDecodeError, ValidationError) as first_error:
        pass

    # Attempt 2: feed the validation error back and ask the LLM to correct itself
    retry_messages = messages + [
        AIMessage(content=raw_output),
        HumanMessage(
            content=(
                "Your previous response was invalid: "
                f"{first_error}\n"
                "Return ONLY a corrected JSON object with exactly the required "
                "fields, categories, priorities, and team mapping described above."
            )
        ),
    ]
    raw_output_retry = llm.invoke(retry_messages).content
    try:
        return parse_and_validate(raw_output_retry).model_dump()
    except (json.JSONDecodeError, ValidationError):
        pass

    # Both attempts failed validation -> safe fallback
    return dict(FALLBACK_RESULT)

## Run (Validated)

In [67]:
sample_ticket_validated = """mala order milali nahi tri display hot ahe delievered"""
result_validated = robust_classify_ticket(sample_ticket_validated)

print(json.dumps(result_validated, indent=4))

{
    "category": "delivery_logistics",
    "priority": "High",
    "assigned_team": "Logistics & Delivery Ops",
    "reasoning": "Order marked as delivered but not received, indicating a serious delivery issue."
}
